# Find the best results

In [1]:
using Pkg
Pkg.activate("../")
include("../scripts/CopepodsNN.jl")
include("../scripts/param.jl")
using NCDatasets
using Dates
using Glob
using DIVAnd
using Statistics
using JupyterFormatter
enable_autoformat()

  Activating project at `~/Projects/EMODnet/EMODnet-Biology/EMODnet-Biology-Interpolation-NN-Copepods`


1-element Vector{Function}:
 format_current_cell (generic function with 1 method)

## Files and directories

In [9]:
varname = "Small_copepods"
# varname = "Large_copepods"

outputdir_optim = joinpath(outputdir, "param_optim/$(varname)")
isdir(outputdir_optim)
expdirlist = filter(x -> isdir(joinpath(outputdir_optim, x)), readdir(outputdir_optim));
@info("Working on $(length(expdirlist)) experiments")
@info("Figures will be saved in $(figdir)")
@info("Files located in $(outputdir_optim)")

[ Info: Working on 975 experiments
[ Info: Figures will be saved in ../product/figures/
[ Info: Files located in ../product/param_optim/Small_copepods


In [10]:
vmin = Dict("Small_copepods" => 0.0, "Large_copepods" => 0.0)
vmax = Dict("Small_copepods" => 1.0, "Large_copepods" => 50.0)

Dict{String, Float64} with 2 entries:
  "Large_copepods" => 50.0
  "Small_copepods" => 1.0

In [11]:
@info(expdirlist[ndirproc]);

[ Info: 20260409T142703


## Find the best analysis

In [12]:
io = open(joinpath(outputdir, "results_$(varname).txt"), "w");
RMSlist = []
allfilelist = []
notfinished = 0
ndirproc = 0
for expdir in expdirlist
    @debug(expdir)
    resfilelist = filter(
        x -> isfile(joinpath(outputdir_optim, expdir, x)),
        readdir(joinpath(outputdir_optim, expdir)),
    )
    ndirproc += 1
    if length(resfilelist) == 7
        filter!(xx -> .!occursin("param", xx), resfilelist)

        for resfile in resfilelist
            _, RMS = get_field(joinpath(outputdir_optim, expdir, resfile), varname)
            write(io, "$(resfile)\t $(RMS)\n")

            push!(RMSlist, RMS)
            push!(allfilelist, resfile)
        end
    else
        @debug("Experiment not completed")
        notfinished += 1
    end
end
close(io);
@info(expdirlist[ndirproc]);

┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
┌ Warning: RMS not written in file
└ @ Main In[4]:11
[ Info: 20260410T124440


In [14]:
minRMS = minimum(RMSlist)
bestfile = allfilelist[argmin(RMSlist)];
@show(minRMS, bestfile)
expdirbest = split(split(bestfile, "_")[3], ".")[1];

minRMS = 0.541
bestfile = "Small_copepods_20260325T184717_bathy.nc"


### Parameters of the best analysis

In [16]:
bestfileparam = "../product/param_optim/$(varname)/$(expdirbest)/$(varname)_$(expdirbest)_params.nc"
ds = NCDataset(bestfileparam)
loss = ds["losses"][:]
@show(ds)
close(ds)

ds = Dataset: ../product/param_optim/Small_copepods/20260325T184717/Small_copepods_20260325T184717_params.nc
Group: /

Dimensions
   epochs = 50

Variables
  losses   (50)
    Datatype:    Float64 (Float64)
    Dimensions:  epochs

Global attributes
  epochs               = 50
  batch_size           = 32
  truth_uncertain      = 0
  enc_nfilter_internal = Int32[10, 20, 30, 40, 50]
  skipconnections      = Int32[2, 3, 4, 5, 6]
  clip_grad            = 5.0
  regularization_L1_beta = 0
  regularization_L2_beta = 0
  save_epochs          = 50
  upsampling_method    = nearest
  probability_skip_for_training = 0.2
  jitter_std_pos       = [5.0, 5.0]
  ntime_win            = 9
  learning_rate        = 0.0039700307349593575
  learning_rate_decay_epoch = Inf
  min_std_err          = 0.006737946999085467
  loss_weights_refine  = 1.0
  auxdata_filenames    = ../data/derived_data/dist_coast_interp_1deg_time.nc
  auxdata_varnames     = dist
  auxdata_errvarname   = dist_error
  savesnapshot        

closed Dataset